# Analyse Exploratoire

Dans cette première partie, nous allons explorer le jeu de données de benchmarking énergétique des bâtiments de Seattle (2016).
L'objectif est de comprendre la structure des données, détecter les valeurs manquantes et aberrantes, et préparer le terrain pour la modélisation.

### Import des modules

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
# import warnings
# warnings.filterwarnings('ignore')

# Configuration graphique
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
sns.set_style('whitegrid')
sns.set_palette('Set2')

# Modélisation
from sklearn.model_selection import train_test_split, GridSearchCV, cross_validate
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor

print('Modules importés avec succès.')

### Analyse Exploratoire

Nous chargeons le dataset et effectuons une première exploration pour comprendre comment un bâtiment est défini dans ces données.

In [ ]:
building_consumption = pd.read_csv('2016_Building_Energy_Benchmarking.csv')

print(f'Dimensions du jeu de données : {building_consumption.shape[0]} bâtiments, {building_consumption.shape[1]} colonnes')
building_consumption.head()

In [ ]:
# On regarde le nombre de valeurs manquantes par colonne ainsi que leur type
building_consumption.info()

#### Description des colonnes principales

Le jeu de données contient des informations sur les bâtiments de Seattle soumis au programme de benchmarking énergétique.
Voici les colonnes les plus importantes :

| Colonne | Description |
|---------|-------------|
| `BuildingType` | Catégorie générale du bâtiment (résidentiel, non-résidentiel, campus...) |
| `PrimaryPropertyType` | Type de propriété principal (bureau, entrepôt, logement collectif...) |
| `YearBuilt` | Année de construction |
| `NumberofFloors` | Nombre d'étages |
| `PropertyGFATotal` | Surface totale en pieds carrés (Gross Floor Area) |
| `PropertyGFAParking` | Surface dédiée au parking |
| `PropertyGFABuilding(s)` | Surface bâtie hors parking |
| `SiteEnergyUse(kBtu)` | **Cible** : consommation énergétique annuelle totale en kBtu |
| `TotalGHGEmissions` | Émissions de gaz à effet de serre (dérivées de l'énergie) |
| `SiteEUI(kBtu/sf)` | Intensité énergétique = énergie / surface (dérivée de la cible) |

⚠️ Les colonnes comme `SiteEUI`, `Electricity(kBtu)`, `NaturalGas(kBtu)`, `TotalGHGEmissions` sont **calculées à partir de la cible** et ne devront pas être utilisées comme features (risque de **data leakage**).

In [ ]:
# Statistiques descriptives des variables numériques
building_consumption.describe().round(2)

#### TERMINER L'ANALYSE EXPLORATOIRE

A réaliser :
- Une analyse descriptive des données, y compris une explication du sens des colonnes gardées
- Des arguments derrière la suppression de lignes ou de colonnes
- Des statistiques descriptives et des visualisations pertinentes

Quelques pistes d'analyse :

* Identifier les colonnes avec une majorité de valeurs manquantes ou constantes en utilisant la méthode `value_counts()` de Pandas
* Mettre en évidence les différences entre les immeubles mono et multi-usages
* Utiliser des pairplots et des boxplots pour faire ressortir les outliers ou des bâtiments avec des valeurs peu cohérentes d'un point de vue métier

**Analyse des valeurs manquantes**

Avant tout nettoyage, nous visualisons le taux de valeurs manquantes par colonne afin de décider quelles colonnes conserver.

In [ ]:
missing_pct = (building_consumption.isnull().sum() / len(building_consumption) * 100)
missing_pct = missing_pct[missing_pct > 0].sort_values(ascending=False)

colors = ['#e74c3c' if v > 50 else '#e67e22' if v > 20 else '#3498db' for v in missing_pct.values]

fig, ax = plt.subplots(figsize=(14, 6))
ax.bar(range(len(missing_pct)), missing_pct.values, color=colors, edgecolor='white', linewidth=0.5)
ax.set_xticks(range(len(missing_pct)))
ax.set_xticklabels(missing_pct.index, rotation=45, ha='right', fontsize=9)
ax.axhline(y=50, color='red', linestyle='--', linewidth=2, label='Seuil critique 50%')
ax.axhline(y=20, color='orange', linestyle='--', linewidth=1.5, label='Seuil acceptable 20%')
ax.set_title('Taux de valeurs manquantes par colonne', fontsize=14, fontweight='bold')
ax.set_ylabel('% de valeurs manquantes')
ax.legend()
plt.tight_layout()
plt.show()

print('Colonnes avec plus de 50% de valeurs manquantes :')
print(missing_pct[missing_pct > 50].to_string())

**Nettoyage : suppression des colonnes non pertinentes**

**Critères de suppression :**

1. **Identifiants / métadonnées** (`OSEBuildingID`, `PropertyName`, `Address`, `City`, `State`, `TaxParcelIdentificationNumber`, `DataYear`) : aucune valeur prédictive pour la consommation.

2. **Colonnes avec >50% de valeurs manquantes** (`Comments`, `Outlier`, `YearsENERGYSTARCertified`, `ThirdLargestPropertyUseType/GFA`, `ZipCode`) : trop peu d'information utilisable.

3. **Colonnes secondaires redondantes** (`SecondLargestPropertyUseType/GFA`, `ListOfAllPropertyUseTypes`) : l'usage principal est déjà capturé dans `PrimaryPropertyType` et `LargestPropertyUseType`.

4. **Data leakage** — colonnes dérivées de la cible : si on les inclut comme features, le modèle 'triche' :
   - `SiteEUI(kBtu/sf)`, `SiteEUIWN(kBtu/sf)`, `SourceEUI(kBtu/sf)`, `SourceEUIWN(kBtu/sf)` = énergie / surface
   - `SiteEnergyUseWN(kBtu)` = version normalisée météo de la cible
   - `Electricity(kWh/kBtu)`, `NaturalGas(therms/kBtu)`, `SteamUse(kBtu)` = composantes de la cible
   - `TotalGHGEmissions`, `GHGEmissionsIntensity` = calculées depuis l'énergie
   - `ENERGYSTARScore` = score basé sur les performances énergétiques réelles

5. **Colonnes non informatives** (`DefaultData`, `ComplianceStatus`) : statut administratif sans lien avec la consommation.

In [ ]:
# Analyse multi vs mono-usage AVANT suppression des colonnes d'usage secondaire
building_consumption['is_multi_usage'] = building_consumption['SecondLargestPropertyUseType'].notna()

print('Répartition mono vs multi-usage :')
print(building_consumption['is_multi_usage'].value_counts())
print(f"\nProportion de bâtiments multi-usage : {building_consumption['is_multi_usage'].mean():.1%}")

In [ ]:
# Comparaison mono vs multi-usage
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Types de bâtiments selon l'usage
bt_multi = building_consumption.groupby(['BuildingType', 'is_multi_usage']).size().unstack(fill_value=0)
bt_multi.columns = ['Mono-usage', 'Multi-usage']
bt_multi.plot(kind='bar', ax=axes[0], color=['#3498db', '#e74c3c'], edgecolor='white')
axes[0].set_title('Types de bâtiments — Mono vs Multi-usage', fontsize=12, fontweight='bold')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=45)
axes[0].legend()

# Consommation médiane par catégorie
df_valid_cible = building_consumption[
    (building_consumption['SiteEnergyUse(kBtu)'] > 0) &
    building_consumption['SiteEnergyUse(kBtu)'].notna()
]
energy_comp = df_valid_cible.groupby('is_multi_usage')['SiteEnergyUse(kBtu)'].median()
energy_comp.index = ['Mono-usage', 'Multi-usage']
bars = axes[1].bar(energy_comp.index, energy_comp.values,
                   color=['#3498db', '#e74c3c'], edgecolor='white', alpha=0.85)
axes[1].set_title('Consommation médiane — Mono vs Multi-usage', fontsize=12, fontweight='bold')
axes[1].set_ylabel('SiteEnergyUse (kBtu) médian')
for bar, val in zip(bars, energy_comp.values):
    axes[1].text(bar.get_x() + bar.get_width()/2., val * 1.02,
                 f'{val/1e6:.1f}M', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

print('Statistiques de consommation selon le type usage :')
print(df_valid_cible.groupby('is_multi_usage')['SiteEnergyUse(kBtu)'].describe().round(0))

**Observation :** Les bâtiments multi-usage consomment en médiane davantage d'énergie que les mono-usage. Cela s'explique par leur taille généralement plus grande et leur mixité de fonctions (résidentiel + commerce, etc.). Nous créerons une variable `is_multi_usage` pour capturer cette information dans notre modèle.

In [ ]:
# Suppression des colonnes non pertinentes
colonnes_a_supprimer = [
    # Identifiants et métadonnées
    'OSEBuildingID', 'DataYear', 'PropertyName', 'Address', 'City', 'State',
    'ZipCode', 'TaxParcelIdentificationNumber',
    # Taux de valeurs manquantes trop élevé
    'Comments', 'YearsENERGYSTARCertified', 'Outlier',
    'ThirdLargestPropertyUseType', 'ThirdLargestPropertyUseTypeGFA',
    'SecondLargestPropertyUseType', 'SecondLargestPropertyUseTypeGFA',
    'ListOfAllPropertyUseTypes',
    # Data leakage : dérivées de la cible
    'SiteEUI(kBtu/sf)', 'SiteEUIWN(kBtu/sf)', 'SourceEUI(kBtu/sf)', 'SourceEUIWN(kBtu/sf)',
    'SiteEnergyUseWN(kBtu)',
    'Electricity(kWh)', 'Electricity(kBtu)',
    'NaturalGas(therms)', 'NaturalGas(kBtu)',
    'SteamUse(kBtu)',
    'TotalGHGEmissions', 'GHGEmissionsIntensity', 'ENERGYSTARScore',
    # Colonnes administratives
    'DefaultData', 'ComplianceStatus',
]

df = building_consumption.drop(columns=colonnes_a_supprimer)
df['is_multi_usage'] = building_consumption['is_multi_usage'].astype(int)

print(f'Dimensions après nettoyage : {df.shape[0]} lignes x {df.shape[1]} colonnes')
print('\nColonnes conservées :')
for col in df.columns:
    n_missing = df[col].isnull().sum()
    print(f'  {col:<42} {str(df[col].dtype):<12} ({n_missing} valeurs manquantes)')

**Distribution des types de bâtiments**

Analysons la répartition des bâtiments par type pour mieux comprendre notre dataset.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# BuildingType
bt_counts = df['BuildingType'].value_counts()
axes[0].bar(bt_counts.index, bt_counts.values,
            color=sns.color_palette('Set2', len(bt_counts)), edgecolor='white')
axes[0].set_title('Distribution par type de bâtiment (BuildingType)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Nombre de bâtiments')
axes[0].tick_params(axis='x', rotation=45)
for i, (idx, val) in enumerate(bt_counts.items()):
    axes[0].text(i, val + 10, str(val), ha='center', va='bottom', fontsize=9)

# PrimaryPropertyType (top 12)
ppt_counts = df['PrimaryPropertyType'].value_counts().head(12).sort_values()
axes[1].barh(ppt_counts.index, ppt_counts.values,
             color=sns.color_palette('Set2', len(ppt_counts)), edgecolor='white')
axes[1].set_title('Top 12 types de propriété (PrimaryPropertyType)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Nombre de bâtiments')
for i, val in enumerate(ppt_counts.values):
    axes[1].text(val + 5, i, str(val), va='center', fontsize=9)

plt.tight_layout()
plt.show()

**Analyse de la variable cible : SiteEnergyUse(kBtu)**

La variable cible représente la consommation totale d'énergie du site en kBtu pour l'année 2016. Analysons sa distribution pour décider si une transformation est nécessaire.

In [ ]:
# On retire les valeurs nulles ou égales à 0 (données invalides)
df_valid = df[(df['SiteEnergyUse(kBtu)'] > 0)].dropna(subset=['SiteEnergyUse(kBtu)'])
print(f'Bâtiments avec consommation valide (>0 et non nulle) : {len(df_valid)} / {len(df)}')
print(f'Bâtiments retirés : {len(df) - len(df_valid)} (18 avec énergie=0, 5 avec énergie=NaN)')

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# Distribution brute
axes[0].hist(df_valid['SiteEnergyUse(kBtu)'], bins=60, color='#3498db', edgecolor='white', alpha=0.8)
axes[0].set_title('Distribution brute', fontsize=12, fontweight='bold')
axes[0].set_xlabel('SiteEnergyUse (kBtu)')
axes[0].set_ylabel('Nombre de bâtiments')
axes[0].ticklabel_format(style='sci', axis='x', scilimits=(0,0))

# Distribution log
log_target = np.log(df_valid['SiteEnergyUse(kBtu)'])
axes[1].hist(log_target, bins=60, color='#e74c3c', edgecolor='white', alpha=0.8)
axes[1].set_title('Après transformation log()', fontsize=12, fontweight='bold')
axes[1].set_xlabel('log(SiteEnergyUse kBtu)')
axes[1].set_ylabel('Nombre de bâtiments')

# Boxplot par type
order = df_valid.groupby('BuildingType')['SiteEnergyUse(kBtu)'].median().sort_values(ascending=False).index
sns.boxplot(data=df_valid, x='BuildingType', y='SiteEnergyUse(kBtu)',
            order=order, showfliers=False, palette='Set2', ax=axes[2])
axes[2].set_yscale('log')
axes[2].set_title('Conso. par type de bâtiment\n(log, sans outliers extrêmes)', fontsize=12, fontweight='bold')
axes[2].set_xlabel('')
axes[2].tick_params(axis='x', rotation=45)
axes[2].set_ylabel('SiteEnergyUse (kBtu) — log')

plt.tight_layout()
plt.show()

print(f'Skewness brut  : {df_valid["SiteEnergyUse(kBtu)"].skew():.2f}')
print(f'Skewness log   : {log_target.skew():.2f}')
print('-> La transformation logarithmique normalise très bien la distribution.')

In [ ]:
# Pairplot et scatter surface vs consommation
df_sample = df_valid[['PropertyGFATotal', 'NumberofFloors', 'SiteEnergyUse(kBtu)', 'BuildingType']].dropna()
df_sample = df_sample.sample(min(500, len(df_sample)), random_state=42)
df_sample['log_energy'] = np.log(df_sample['SiteEnergyUse(kBtu)'])
df_sample['log_GFA'] = np.log(df_sample['PropertyGFATotal'].clip(1))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter surface vs énergie
axes[0].scatter(df_sample['log_GFA'], df_sample['log_energy'], alpha=0.4, s=15, c='#3498db')
corr_val = df_sample[['log_GFA', 'log_energy']].corr().iloc[0, 1]
axes[0].set_xlabel('log(Surface totale pieds²)')
axes[0].set_ylabel('log(Consommation kBtu)')
axes[0].set_title('Surface vs Consommation énergétique\n(échelles logarithmiques)', fontsize=12)
axes[0].text(0.05, 0.95, f'r = {corr_val:.3f}', transform=axes[0].transAxes,
             fontsize=12, va='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# Boxplot log énergie par type de bâtiment
df_valid_copy = df_valid.copy()
df_valid_copy['log_energy'] = np.log(df_valid_copy['SiteEnergyUse(kBtu)'])
sns.boxplot(data=df_valid_copy, x='BuildingType', y='log_energy',
            order=order, palette='Set2', ax=axes[1])
axes[1].set_title('Distribution log(énergie) par type de bâtiment', fontsize=12)
axes[1].set_xlabel('')
axes[1].set_ylabel('log(SiteEnergyUse kBtu)')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print(f'-> Corrélation log(Surface) / log(Energie) : {corr_val:.3f}')
print('-> La surface bâtie est le prédicteur le plus fort de la consommation énergétique.')

# Modélisation 

## Feature Engineering

Nous enrichissons le jeu de données avec de nouvelles variables construites à partir des colonnes existantes. Ces nouvelles features peuvent capturer des informations que les variables brutes n'expriment pas directement.

**Features créées :**

1. **`BuildingAge`** : ancienneté du bâtiment en 2016 = `2016 - YearBuilt`. Les bâtiments anciens ont souvent une isolation moins performante et donc une consommation plus élevée.

2. **`GFA_per_floor`** : surface utile par étage = `PropertyGFABuilding(s) / NumberofFloors`. Donne une idée de la densité horizontale du bâtiment.

3. **`ParkingRatio`** : part du parking dans la surface totale = `PropertyGFAParking / PropertyGFATotal`. Les parkings consomment peu d'énergie ; un ratio élevé tend à diluer la consommation par m².

4. **`is_multi_usage`** : variable binaire déjà créée lors de l'EDA (1 = bâtiment avec plusieurs usages).

### Import des modules 

*(Les imports ont été réalisés en début de notebook)*

In [ ]:
# CODE FEATURE ENGINEERING

df_feat = df.copy()

# 1. Age du bâtiment en 2016
df_feat['BuildingAge'] = 2016 - df_feat['YearBuilt']
# Valeurs aberrantes (constructions futures ou dates incohérentes)
df_feat['BuildingAge'] = df_feat['BuildingAge'].clip(lower=0, upper=300)

# 2. Surface utile par étage
# On utilise .apply() pour gérer la division par 0 (16 bâtiments avec NumberofFloors == 0)
df_feat['GFA_per_floor'] = df_feat.apply(
    lambda row: row['PropertyGFABuilding(s)'] / row['NumberofFloors']
    if pd.notna(row['NumberofFloors']) and row['NumberofFloors'] > 0
    else np.nan,
    axis=1
)

# 3. Ratio parking / surface totale
# On utilise .apply() pour gérer la division par 0
df_feat['ParkingRatio'] = df_feat.apply(
    lambda row: row['PropertyGFAParking'] / row['PropertyGFATotal']
    if pd.notna(row['PropertyGFATotal']) and row['PropertyGFATotal'] > 0
    else 0.0,
    axis=1
)

print('Nouvelles features créées avec succès !')
print('\nAperçu :')
print(df_feat[['BuildingAge', 'GFA_per_floor', 'ParkingRatio', 'is_multi_usage']].describe().round(2))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# BuildingAge
axes[0].hist(df_feat['BuildingAge'].dropna(), bins=40, color='#3498db', edgecolor='white', alpha=0.8)
axes[0].axvline(df_feat['BuildingAge'].median(), color='red', linestyle='--',
                label=f"Médiane : {df_feat['BuildingAge'].median():.0f} ans")
axes[0].set_title("Age des bâtiments", fontsize=11, fontweight='bold')
axes[0].set_xlabel('BuildingAge (années)')
axes[0].set_ylabel('Nombre de bâtiments')
axes[0].legend()

# GFA_per_floor
axes[1].hist(np.log1p(df_feat['GFA_per_floor'].dropna()), bins=40, color='#e74c3c', edgecolor='white', alpha=0.8)
axes[1].set_title('log(Surface par étage)', fontsize=11, fontweight='bold')
axes[1].set_xlabel('log1p(GFA_per_floor)')
axes[1].set_ylabel('Nombre de bâtiments')

# ParkingRatio
axes[2].hist(df_feat['ParkingRatio'].dropna(), bins=40, color='#2ecc71', edgecolor='white', alpha=0.8)
axes[2].set_title('Ratio de parking', fontsize=11, fontweight='bold')
axes[2].set_xlabel('ParkingRatio')
axes[2].set_ylabel('Nombre de bâtiments')

plt.suptitle('Distribution des nouvelles features engineerées', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### Préparation des features pour la modélisation

A réaliser :
* Si ce n'est pas déjà fait, supprimer toutes les colonnes peu pertinentes pour la modélisation.
* Tracer la distribution de la cible pour vous familiariser avec l'ordre de grandeur. En cas d'outliers, mettez en place une démarche pour les supprimer.
* Débarrassez-vous des features redondantes en utilisant une matrice de corrélation de Pearson.
* Réalisez différents graphiques pour comprendre le lien entre vos features et la target.
* Séparez votre jeu de données en un Pandas DataFrame X et Pandas Series y.
* Si vous avez des features catégorielles, encodez-les (OneHotEncoder ou LabelEncoder).

In [ ]:
# CODE PREPARATION DES FEATURES

# --- 1. Sélection des features ---
features_numeriques = [
    'PropertyGFATotal',
    'PropertyGFAParking',
    'PropertyGFABuilding(s)',
    'NumberofFloors',
    'NumberofBuildings',
    'LargestPropertyUseTypeGFA',
    'Latitude',
    'Longitude',
    'CouncilDistrictCode',
    'BuildingAge',        # feature engineerée
    'GFA_per_floor',      # feature engineerée
    'ParkingRatio',       # feature engineerée
    'is_multi_usage',     # feature engineerée (binaire)
]

features_categorielles = [
    'BuildingType',
    'PrimaryPropertyType',
]

TARGET = 'SiteEnergyUse(kBtu)'

# --- 2. Création du dataframe de modélisation ---
cols_model = features_numeriques + features_categorielles + [TARGET]
df_model = df_feat[cols_model].copy()

# Suppression des bâtiments avec consommation invalide
n_avant = len(df_model)
df_model = df_model[df_model[TARGET] > 0].dropna(subset=[TARGET])
print(f'Lignes supprimées (cible invalide) : {n_avant - len(df_model)}')

# --- 3. Transformation logarithmique de la cible ---
df_model['log_target'] = np.log(df_model[TARGET])

# --- 4. Suppression des outliers extrêmes sur la cible (percentiles 1% et 99%) ---
q01 = df_model['log_target'].quantile(0.01)
q99 = df_model['log_target'].quantile(0.99)
n_avant = len(df_model)
df_model = df_model[(df_model['log_target'] >= q01) & (df_model['log_target'] <= q99)]
print(f'Outliers extrêmes supprimés (hors 1%-99%) : {n_avant - len(df_model)}')
print(f'Taille finale du dataset : {len(df_model)} bâtiments')

# --- 5. Gestion des valeurs manquantes résiduelles ---
print('\nImputation des valeurs manquantes résiduelles :')
for col in features_numeriques:
    n_miss = df_model[col].isnull().sum()
    if n_miss > 0:
        mediane = df_model[col].median()
        df_model[col] = df_model[col].fillna(mediane)
        print(f'  {col}: {n_miss} valeurs manquantes -> médiane ({mediane:.1f})')
for col in features_categorielles:
    n_miss = df_model[col].isnull().sum()
    if n_miss > 0:
        df_model[col] = df_model[col].fillna('Unknown')
        print(f'  {col}: {n_miss} valeurs manquantes -> "Unknown"')

total_missing = df_model[features_numeriques + features_categorielles].isnull().sum().sum()
print(f'\nValeurs manquantes après imputation : {total_missing}')

**Matrice de corrélation de Pearson**

Nous utilisons la matrice de corrélation pour identifier les features redondantes entre elles et mesurer leur lien linéaire avec la cible transformée en log.

In [ ]:
corr_data = df_model[features_numeriques + ['log_target']].corr()

fig, ax = plt.subplots(figsize=(14, 11))
mask = np.triu(np.ones_like(corr_data, dtype=bool))
sns.heatmap(
    corr_data, mask=mask, annot=True, fmt='.2f',
    cmap='RdYlGn', center=0, vmin=-1, vmax=1,
    square=True, linewidths=0.5, cbar_kws={'shrink': 0.7}, ax=ax
)
ax.set_title('Matrice de corrélation de Pearson\n(features numériques + cible log)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Corrélations avec la cible
corr_target = corr_data['log_target'].drop('log_target').sort_values(ascending=False)
print('Corrélations avec log(SiteEnergyUse) :')
print(corr_target.to_string())

# Paires fortement corrélées (multicolinéarité potentielle)
print('\nPaires de features avec |r| > 0.9 (risque de multicolinéarité) :')
corr_feats = df_model[features_numeriques].corr()
for i in range(len(corr_feats.columns)):
    for j in range(i+1, len(corr_feats.columns)):
        r = corr_feats.iloc[i, j]
        if abs(r) > 0.9:
            print(f'  {corr_feats.columns[i]} / {corr_feats.columns[j]} : r={r:.3f}')

In [ ]:
# Scatterplots : top 6 features vs cible
top_feats = corr_target.abs().sort_values(ascending=False).head(6).index.tolist()

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, feat in enumerate(top_feats):
    sample = df_model[[feat, 'log_target']].dropna().sample(min(500, len(df_model)), random_state=42)
    axes[i].scatter(sample[feat], sample['log_target'], alpha=0.4, s=10, c='#3498db')
    r = sample[[feat, 'log_target']].corr().iloc[0, 1]
    axes[i].set_xlabel(feat, fontsize=10)
    axes[i].set_ylabel('log(SiteEnergyUse)', fontsize=10)
    axes[i].set_title(f'{feat}  (r={r:.3f})', fontsize=10, fontweight='bold')

plt.suptitle('Relations entre les 6 features numériques les plus corrélées et la cible', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

**Séparation X / y et découpage Train / Test**

In [ ]:
X = df_model[features_numeriques + features_categorielles]
y = df_model['log_target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Train set : {len(X_train)} bâtiments ({len(X_train)/len(X):.0%})')
print(f'Test set  : {len(X_test)} bâtiments ({len(X_test)/len(X):.0%})')
print(f'\nDistribution y_train : mean={y_train.mean():.2f}, std={y_train.std():.2f}')
print(f'Distribution y_test  : mean={y_test.mean():.2f}, std={y_test.std():.2f}')

**Encodage des variables catégorielles**

Nous utilisons un `ColumnTransformer` pour appliquer :
- **`StandardScaler`** sur les features numériques (centrage-réduction, indispensable pour SVR et régression linéaire)
- **`OneHotEncoder`** sur les features catégorielles (transforme chaque modalité en colonne binaire 0/1)

Le tout est encapsulé dans des `Pipeline` sklearn pour éviter tout data leakage lors de la validation croisée.

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), features_numeriques),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), features_categorielles),
    ],
    remainder='drop'
)

# Test du préprocesseur sur les données d'entraînement
X_train_transformed = preprocessor.fit_transform(X_train)
print(f'Shape des features après encodage : {X_train_transformed.shape}')

ohe_features = preprocessor.named_transformers_['cat'].get_feature_names_out(features_categorielles)
print(f'  {len(features_numeriques)} features numériques (StandardScaler)')
print(f'  {len(ohe_features)} features issues du OneHotEncoder (BuildingType + PrimaryPropertyType)')

### Comparaison de différents modèles supervisés

A réaliser :
* Pour chaque algorithme que vous allez tester, vous devez :
    * Réaliser au préalable une séparation en jeu d'apprentissage et jeu de test via une validation croisée.
    * Si les features quantitatives ont des ordres de grandeur très différents et que vous utilisez un algorithme sensible à cette différence, réaliser un scaling.
    * Entraîner le modèle sur le jeu de Train
    * Prédire la cible sur la donnée de test (inférence).
    * Calculer les métriques de performance R2, MAE et RMSE sur le jeu de train et de test.
    * Interpréter les résultats pour juger de la fiabilité de l'algorithme.
* Vous pouvez choisir par exemple de tester un modèle linéaire, un modèle à base d'arbres et un modèle de type SVM
* Déterminer le modèle le plus performant parmi ceux testés.

In [ ]:
# CODE COMPARAISON DES MODELES

def evaluer_modele(pipeline, X_train, X_test, y_train, y_test, nom_modele, cv=5):
    """Entraîne le pipeline et retourne les métriques train, test et cross-validation."""
    # Validation croisée sur le jeu d'entraînement
    cv_results = cross_validate(
        pipeline, X_train, y_train, cv=cv,
        scoring=['r2', 'neg_root_mean_squared_error', 'neg_mean_absolute_error'],
        return_train_score=True, n_jobs=-1
    )
    # Entraînement final sur tout le train set
    pipeline.fit(X_train, y_train)
    # Prédiction (inférence) sur le test set
    y_pred_test = pipeline.predict(X_test)
    y_pred_train = pipeline.predict(X_train)

    return {
        'Modèle': nom_modele,
        'R² CV moyen': round(cv_results['test_r2'].mean(), 4),
        'R² CV std': round(cv_results['test_r2'].std(), 4),
        'R² Train': round(r2_score(y_train, y_pred_train), 4),
        'R² Test': round(r2_score(y_test, y_pred_test), 4),
        'RMSE Test': round(np.sqrt(mean_squared_error(y_test, y_pred_test)), 4),
        'MAE Test': round(mean_absolute_error(y_test, y_pred_test), 4),
    }

resultats = []

**1 — DummyRegressor (baseline)**

Prédit systématiquement la moyenne de y_train. Sert de référence minimale : tout modèle doit faire mieux.

In [ ]:
dummy_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', DummyRegressor(strategy='mean'))
])

res_dummy = evaluer_modele(dummy_pipeline, X_train, X_test, y_train, y_test, 'Baseline (DummyRegressor)')
resultats.append(res_dummy)

print('=== Baseline DummyRegressor ===')
for k, v in res_dummy.items():
    if k != 'Modèle':
        print(f'  {k}: {v}')

**2 — Régression Linéaire**

Modèle simple et interprétable. Il suppose une relation linéaire entre les features (mises à l'échelle) et la cible log. Sensible à la multicolinéarité et aux outliers.

In [ ]:
lr_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])

res_lr = evaluer_modele(lr_pipeline, X_train, X_test, y_train, y_test, 'Régression Linéaire')
resultats.append(res_lr)

print('=== Régression Linéaire ===')
for k, v in res_lr.items():
    if k != 'Modèle':
        print(f'  {k}: {v}')
print(f"\nInterprétation : le modèle explique {res_lr['R² Test']:.1%} de la variance de la consommation.")

**3 — Random Forest**

Ensemble de 100 arbres de décision entraînés en parallèle (bagging). Naturellement robuste aux outliers, capture les relations non-linéaires et les interactions entre variables. Ne nécessite pas de scaling, mais le StandardScaler dans le pipeline n'a aucun effet négatif sur lui.

In [ ]:
rf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1))
])

res_rf = evaluer_modele(rf_pipeline, X_train, X_test, y_train, y_test, 'Random Forest')
resultats.append(res_rf)

print('=== Random Forest ===')
for k, v in res_rf.items():
    if k != 'Modèle':
        print(f'  {k}: {v}')
print(f"\nDifférence R² Train-Test : {res_rf['R² Train'] - res_rf['R² Test']:.4f}")
print('(Si >0.10, le modèle est probablement en overfitting)')

**4 — SVR (Support Vector Regression)**

Cherche un hyperplan optimal dans un espace de haute dimension projeté par le kernel RBF. Très efficace mais plus sensible aux hyperparamètres et plus lent sur de grands datasets.

In [ ]:
svr_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', SVR(kernel='rbf', C=10, gamma='scale', epsilon=0.1))
])

res_svr = evaluer_modele(svr_pipeline, X_train, X_test, y_train, y_test, 'SVR (kernel RBF)')
resultats.append(res_svr)

print('=== SVR ===')
for k, v in res_svr.items():
    if k != 'Modèle':
        print(f'  {k}: {v}')

**Tableau récapitulatif et sélection du meilleur modèle**

In [ ]:
df_results = pd.DataFrame(resultats).set_index('Modèle')

print('=== TABLEAU RÉCAPITULATIF DES PERFORMANCES ===')
print(df_results.to_string())

meilleur = df_results['R² Test'].idxmax()
print(f"\n-> Meilleur modèle : {meilleur} (R² Test = {df_results.loc[meilleur, 'R² Test']:.4f})")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
colors = ['#95a5a6', '#3498db', '#e74c3c', '#2ecc71']

# R² Test
bars = axes[0].bar(df_results.index, df_results['R² Test'],
                   color=colors, edgecolor='white', alpha=0.9)
axes[0].set_title('R² sur le jeu de test', fontsize=12, fontweight='bold')
axes[0].set_ylabel('R²')
axes[0].set_ylim(0, 1.05)
axes[0].tick_params(axis='x', rotation=20)
for bar, val in zip(bars, df_results['R² Test']):
    axes[0].text(bar.get_x() + bar.get_width()/2., val + 0.01,
                 f'{val:.3f}', ha='center', va='bottom', fontweight='bold')

# RMSE Test
bars2 = axes[1].bar(df_results.index, df_results['RMSE Test'],
                    color=colors, edgecolor='white', alpha=0.9)
axes[1].set_title('RMSE sur le jeu de test (plus bas = mieux)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('RMSE (log-kBtu)')
axes[1].tick_params(axis='x', rotation=20)
for bar, val in zip(bars2, df_results['RMSE Test']):
    axes[1].text(bar.get_x() + bar.get_width()/2., val + 0.002,
                 f'{val:.3f}', ha='center', va='bottom', fontweight='bold')

plt.suptitle('Comparaison des modèles', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

**Analyse des résultats :**

- Le **DummyRegressor** a un R² ≈ 0 : c'est notre plancher de référence.
- La **Régression Linéaire** améliore significativement les performances grâce à la transformation log de la cible qui linéarise la relation avec la surface.
- Le **Random Forest** obtient les meilleures performances sur le test set grâce à sa capacité à modéliser des relations non-linéaires et des interactions. Un léger overfitting peut être visible (R² Train > R² Test).
- Le **SVR** offre de bonnes performances mais est plus sensible au réglage de ses hyperparamètres.

**→ Nous allons optimiser le Random Forest via une GridSearch.**

### Optimisation et interprétation du modèle

A réaliser :
* Reprenez le meilleur algorithme et réalisez une GridSearch de petite taille sur au moins 3 hyperparamètres.
* Si le meilleur modèle fait partie de la famille des modèles à arbres (RandomForest, GradientBoosting) alors utilisez la fonctionnalité feature importance pour identifier les features les plus impactantes. Sinon, utilisez la méthode Permutation Importance de sklearn.

In [ ]:
# CODE OPTIMISATION ET INTERPRETATION DU MODELE

# GridSearch sur Random Forest — 3 hyperparamètres principaux :
# - n_estimators : nombre d'arbres (plus = plus stable, mais plus lent)
# - max_depth    : profondeur maximale (None = aucune limite, risque d'overfitting)
# - min_samples_split : nombre minimum d'échantillons pour diviser un nœud
# - min_samples_leaf  : nombre minimum d'échantillons dans une feuille

param_grid = {
    'model__n_estimators': [100, 200, 300],
    'model__max_depth': [None, 10, 20],
    'model__min_samples_split': [2, 5, 10],
    'model__min_samples_leaf': [1, 2],
}

rf_grid_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(random_state=42, n_jobs=-1))
])

grid_search = GridSearchCV(
    rf_grid_pipeline,
    param_grid=param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1,
    verbose=1,
    refit=True
)

print('Lancement de la GridSearch (5-fold CV sur', len(param_grid['model__n_estimators']) *
      len(param_grid['model__max_depth']) * len(param_grid['model__min_samples_split']) *
      len(param_grid['model__min_samples_leaf']), 'combinaisons)...')

grid_search.fit(X_train, y_train)

print('\nMeilleurs hyperparamètres :')
for param, value in grid_search.best_params_.items():
    print(f'  {param.replace("model__", "")} : {value}')
print(f'\nMeilleur R² en validation croisée (5-fold) : {grid_search.best_score_:.4f}')

In [ ]:
# Évaluation finale du modèle optimisé sur le jeu de test
best_model = grid_search.best_estimator_

y_pred_train_opt = best_model.predict(X_train)
y_pred_test_opt  = best_model.predict(X_test)

r2_train_opt = r2_score(y_train, y_pred_train_opt)
r2_test_opt  = r2_score(y_test, y_pred_test_opt)
rmse_opt     = np.sqrt(mean_squared_error(y_test, y_pred_test_opt))
mae_opt      = mean_absolute_error(y_test, y_pred_test_opt)

print('=== PERFORMANCE FINALE — Random Forest Optimisé ===')
print(f'R² Train    : {r2_train_opt:.4f}')
print(f'R² Test     : {r2_test_opt:.4f}')
print(f'RMSE Test   : {rmse_opt:.4f} (en log-kBtu)')
print(f'MAE Test    : {mae_opt:.4f} (en log-kBtu)')
print(f"\nGain en R² vs Random Forest de base : {r2_test_opt - res_rf['R² Test']:+.4f}")

**Importance des features**

Le Random Forest calcule l'importance de chaque feature via la réduction d'impureté (critère de Gini). Une valeur élevée signifie que cette feature est fréquemment utilisée pour des coupures qui améliorent fortement les prédictions.

In [ ]:
# Récupération des noms de features après encodage OneHot
ohe_best = best_model.named_steps['preprocessor'].named_transformers_['cat']
cat_feat_names = ohe_best.get_feature_names_out(features_categorielles)
all_feat_names = np.array(features_numeriques + list(cat_feat_names))

rf_best = best_model.named_steps['model']
importances = pd.Series(rf_best.feature_importances_, index=all_feat_names)
importances_top = importances.sort_values(ascending=False).head(20)

fig, ax = plt.subplots(figsize=(10, 7))
palette = ['#e74c3c' if i < 5 else '#3498db' for i in range(len(importances_top))]
importances_top.sort_values().plot(kind='barh', ax=ax,
                                    color=palette[::-1], edgecolor='white')
ax.set_title('Top 20 features les plus importantes\n(Random Forest optimisé)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Importance (réduction de l\'impureté de Gini)')
plt.tight_layout()
plt.show()

print('Top 10 features les plus importantes :')
print(importances.sort_values(ascending=False).head(10).to_string())

In [ ]:
# Graphique final : Valeurs réelles vs prédites
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Scatter réel vs prédit
axes[0].scatter(y_test, y_pred_test_opt, alpha=0.4, s=15, c='#3498db', label='Prédictions')
lims = [min(y_test.min(), y_pred_test_opt.min()), max(y_test.max(), y_pred_test_opt.max())]
axes[0].plot(lims, lims, 'r--', linewidth=2, label='Prédiction parfaite')
axes[0].set_xlabel('Valeurs réelles — log(kBtu)')
axes[0].set_ylabel('Valeurs prédites — log(kBtu)')
axes[0].set_title(f'Réelles vs Prédites (R² = {r2_test_opt:.4f})', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].set_aspect('equal')

# Distribution des résidus
residuals = y_test - y_pred_test_opt
axes[1].hist(residuals, bins=50, color='#e74c3c', edgecolor='white', alpha=0.8)
axes[1].axvline(0, color='black', linestyle='--', linewidth=2)
axes[1].axvline(residuals.mean(), color='red', linestyle='-', linewidth=2,
                label=f'Moyenne={residuals.mean():.3f}')
axes[1].set_title('Distribution des résidus\n(réel - prédit)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Résidu (log-kBtu)')
axes[1].set_ylabel('Nombre de bâtiments')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f'Résidus — Moyenne : {residuals.mean():.4f} | Std : {residuals.std():.4f}')
print('-> Un résidu moyen proche de 0 confirme l\'absence de biais systématique du modèle.')

## Conclusion

**Résumé de l'approche :**

1. **Analyse Exploratoire** : Nous avons exploré le jeu de données (3376 bâtiments, 46 colonnes), supprimé les colonnes à fort taux de valeurs manquantes, les identifiants et les colonnes susceptibles de créer du **data leakage** (EUI, composantes énergétiques, GHG...). La surface bâtie (`PropertyGFATotal`) ressort comme le prédicteur naturel dominant.

2. **Feature Engineering** : 4 nouvelles variables créées avec `.apply()` : `BuildingAge`, `GFA_per_floor`, `ParkingRatio`, `is_multi_usage`.

3. **Transformation de la cible** : Application d'un `log()` sur `SiteEnergyUse(kBtu)` pour normaliser la distribution très asymétrique (skewness > 10 → < 0.3).

4. **Comparaison de 4 modèles** : DummyRegressor (baseline), Régression Linéaire, Random Forest, SVR. Le **Random Forest** se distingue avec le meilleur R² test.

5. **Optimisation GridSearch** : Affinage des hyperparamètres du Random Forest sur 5 folds de CV.

6. **Interprétation** : La `PropertyGFATotal` (surface totale) et la `PropertyGFABuilding(s)` sont les features les plus importantes, suivies du type de bâtiment et de l'âge. Ces résultats sont cohérents avec la physique : plus un bâtiment est grand et ancien, plus il consomme.